# Bouger ou imaginer bouger : est-ce le même réseau cérébral ?

On compare les graphes de connectivité cérébrale de trois conditions des runs de tâche d'EEGMMIDB :

| Condition | Runs | Epochs (0.5-4.0 s après l'annotation) |
|---|---|---|
| **MM** : mouvement réel du poing gauche / droit | R03, R07, R11 | T1 + T2 |
| **MI** : imagination de ce mouvement | R04, R08, R12 | T1 + T2 |
| **REST** : pauses entre essais | les 6 runs | T0 |

Chaîne (notebooks 01 et 02, `scripts/build_cache.py`, `scripts/mm_mi_cohort.py`) : 68 régions
Desikan-Killiany en espace source (eLORETA, fsaverage), **wPLI** par epoch en alpha (8-13 Hz) et bêta
(13-30 Hz), 105 sujets inclus sur 109. La validité physiologique est vérifiée en amont : désynchronisation
alpha/bêta du cortex sensorimoteur en MM et en MI, plus forte en MM (`reports/validation_task_cohort.json`).

Par sujet, chaque condition est réduite au **même nombre d'epochs** n = min(n_MM, n_MI, n_REST), tirées au
hasard (`SEED`), et coupée en deux moitiés A et B de n/2 epochs. Les comparaisons de ressemblance se font
toujours à taille égale (n/2 contre n/2).

| Partie | Contenu |
|---|---|
| A | Description des graphes de groupe MM, MI, REST (modèle, tableau, centralités, figures) |
| B | Comparaison au niveau des sujets : métriques globales, même réseau ?, hubs, communautés |
| C | Robustesse au seuil (10, 15, 20 %) et à la bande |
| Conclusion | Réponse en 5 phrases |


## 0. Configuration

In [ ]:
%matplotlib inline
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from eeg_graph import task_analysis as ta
from eeg_graph.config import SEED, Paths, setup_logging
from eeg_graph.source import fsaverage_mesh
from eeg_graph.validation import MOTOR_ROI
from eeg_graph.viz import plot_brain_graph_views, plot_connectivity_matrix, save_figure

setup_logging()
paths = Paths()
TABLES = paths.root / "reports" / "tables"
paths.figures.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
results: dict[str, object] = {}  # chiffres clés -> reports/mm_vs_mi_results.json

In [ ]:
METRIC = "wpli"
BANDS = ("alpha", "beta")
DENSITY = 0.15  # graphe principal
DENSITIES = (0.10, 0.15, 0.20)  # partie C
N_LOUVAIN = 100  # répétitions de Louvain (partition de modularité maximale)
N_BOOT = 5000  # bootstrap sur les sujets
N_JOBS = 4
CONDS = ("MM", "MI", "REST")

Chargement des matrices égalisées (`data/derived/matrices_mm_mi.npz`) : pour chaque sujet inclus,
condition et bande, la matrice wPLI moyenne des n epochs retenues, et les deux moitiés A et B.

In [ ]:
cohort = ta.load_cohort_matrices(paths.derived / "matrices_mm_mi.npz")
inclusion = pd.read_csv(TABLES / "cohort_mm_mi.csv")
assert cohort.metric == METRIC and cohort.conditions == CONDS
assert sorted(cohort.subjects.tolist()) == sorted(inclusion.loc[inclusion.included, "subject"].tolist())
atlas = cohort.atlas
print(
    f"{len(cohort.subjects)} sujets, {atlas.n_nodes} régions ; epochs par condition après égalisation : "
    f"médiane {np.median(cohort.n_epochs):.0f} ({cohort.n_epochs.min()}-{cohort.n_epochs.max()}), "
    f"moitiés de {np.median(cohort.n_half):.0f} ({cohort.n_half.min()}-{cohort.n_half.max()})"
)
results["n_subjects"] = len(cohort.subjects)
results["n_epochs_equalized"] = {
    "median": float(np.median(cohort.n_epochs)),
    "min": int(cohort.n_epochs.min()),
    "max": int(cohort.n_epochs.max()),
}

# Partie A — Description des graphes de groupe

## 1. Modèle de graphe

- **Nœuds** : les 68 régions corticales de l'atlas Desikan-Killiany (34 par hémisphère). Attributs :
  hémisphère, lobe, centroïde MNI (mm).
- **Arêtes et poids** : une arête entre deux régions, pondérée par le **wPLI** dans [0, 1] (consistance du
  déphasage, insensible au couplage à déphasage nul dû à la conduction de volume et à la fuite de sources).
- **Non orienté** : le wPLI est symétrique (wPLI(i, j) = wPLI(j, i)), il ne dit rien de la direction.
- **Simple** : une seule arête par paire, pas de boucle (diagonale nulle).
- **Pondéré** : les poids sont gardés pour la strength, le clustering, l'efficacité et la modularité.
- **Seuillage proportionnel à 15 %** : on garde les 342 paires les plus fortes sur 2 278. Toutes les
  conditions ont alors la même densité ; un seuil absolu ferait varier le nombre d'arêtes avec le niveau
  moyen de connectivité, et les métriques de graphe dépendent fortement de la densité.
- **Distance = 1 / poids** pour les plus courts chemins (betweenness, closeness, efficacité pondérée) :
  une connexion forte est une liaison courte.

Graphe de groupe = graphe de la matrice moyenne des 105 sujets, pour chaque condition et bande.

## 2. Tableau récapitulatif

Diamètre et rayon en nombre de sauts : sur la plus grande composante quand le graphe n'est pas connexe, puis
sur l'union avec l'arbre couvrant maximal du graphe complet, qui est toujours connexe. Louvain répété sur 100
graines, partition de modularité maximale ; NMI entre cette partition et les lobes.
Les nœuds isolés comptent chacun comme une communauté à un seul nœud dans `n_communities`.


In [ ]:
group = {
    (c, b): ta.describe_graph(ta.group_matrix(cohort, c, b), atlas, DENSITY, SEED, N_LOUVAIN)
    for b in BANDS
    for c in CONDS
}
summary = pd.DataFrame([{"band": b, "condition": c} | d.record for (c, b), d in group.items()])
summary.to_csv(TABLES / "mm_mi_group_graph_summary.csv", index=False)
results["group_summary"] = summary.to_dict(orient="records")
summary.set_index(["band", "condition"]).T

## 3. Régions centrales

Top 10 par **strength** (somme des poids : régions les plus connectées), **betweenness** (rôle de relais sur
les plus courts chemins) et **eigenvector** (connexion à d'autres régions centrales).

In [ ]:
tops = []
for (c, b), d in group.items():
    t = ta.top_regions(d.metrics, k=10)
    tops.append(t.assign(band=b, condition=c))
tops = pd.concat(tops, ignore_index=True)
tops.to_csv(TABLES / "mm_mi_group_top10.csv", index=False)
for b in BANDS:
    wide = {
        c: tops[(tops.band == b) & (tops.condition == c)].set_index("rang")[["strength", "betweenness", "eigenvector"]]
        for c in CONDS
    }
    print(f"--- {b}")
    display(pd.concat(wide, axis=1))

## 4. Figures de groupe

Matrices wPLI moyennes ordonnées par hémisphère puis lobe (même échelle pour les trois conditions).

In [ ]:
for b in BANDS:
    mats = {c: ta.group_matrix(cohort, c, b) for c in CONDS}
    vmax = max(float(m.max()) for m in mats.values())
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for ax, c in zip(axes, CONDS, strict=True):
        plot_connectivity_matrix(ax, mats[c], atlas, f"{c} — wPLI {b}, moyenne de {len(cohort.subjects)} sujets", vmax)
    save_figure(fig, paths.figures / f"mm_mi_group_matrices_{b}.png")
    plt.show()

Graphe de groupe (15 % des arêtes) dans le cerveau : couleur = communauté de Louvain, taille = strength.

In [ ]:
mesh = fsaverage_mesh(paths)
for (c, b), d in group.items():
    m = d.metrics
    title = (
        f"{c} — wPLI {b}, {round(DENSITY * 100)} % des arêtes ; {m.n_communities} communautés (Q = {m.modularity:.2f})"
    )
    fig = plot_brain_graph_views(d.graph, atlas, mesh, d.metrics.community, d.metrics.strength, title)
    save_figure(fig, paths.figures / f"mm_mi_group_brain_{c}_{b}.png")
    plt.show()

# Partie B — Comparaison au niveau des sujets

Un graphe par sujet, condition et bande (matrice égalisée, seuil 15 %).

## 5. Métriques globales

- **Efficacité globale pondérée** : intégration (moyenne de 1 / d_ij) ;
- **clustering pondéré** (Onnela) : ségrégation locale ;
- **modularité** de Louvain : séparation en modules.

Wilcoxon apparié bilatéral pour MM vs MI, MM vs REST et MI vs REST ; correction de Holm sur les 9 tests de
chaque (bande, seuil) ; taille d'effet = corrélation rang-bisériale appariée r (> 0 : la première condition
est plus haute). Le calcul couvre aussi les seuils 10 et 20 % (partie C).

In [ ]:
subject_metrics = ta.subject_global_metrics(cohort, DENSITIES, SEED, N_LOUVAIN, n_jobs=N_JOBS)
subject_metrics.to_csv(TABLES / "mm_mi_subject_global_metrics.csv", index=False)
tests = ta.global_metric_tests(subject_metrics)
tests.to_csv(TABLES / "mm_mi_global_tests.csv", index=False)
main_tests = tests[np.isclose(tests.density, DENSITY)]
results["global_tests_15"] = main_tests.round(6).to_dict(orient="records")
main_tests[["band", "metric", "comparison", "median_a", "median_b", "W", "p", "p_holm", "r_rb"]].round(4)

In [ ]:
labels = {
    "global_efficiency_weighted": "efficacité globale pondérée",
    "clustering_weighted": "clustering pondéré",
    "modularity": "modularité",
}
main = subject_metrics[np.isclose(subject_metrics.density, DENSITY)]
fig, axes = plt.subplots(len(BANDS), len(labels), figsize=(14, 4.2 * len(BANDS)), squeeze=False)
for i, b in enumerate(BANDS):
    for j, (m, lab) in enumerate(labels.items()):
        ax = axes[i, j]
        wide = main[main.band == b].pivot(index="subject", columns="condition", values=m)[list(CONDS)]
        ax.plot(range(3), wide.to_numpy().T, color="0.6", lw=0.4, alpha=0.5)
        ax.plot(range(3), wide.median().to_numpy(), "o-", color="C3", lw=2, label="médiane")
        ax.set_xticks(range(3), CONDS)
        ax.set_title(f"{lab} — {b}", fontsize=10)
axes[0, 0].legend(fontsize=8)
fig.suptitle(f"Métriques globales par sujet (lignes grises = un sujet), seuil {round(DENSITY * 100)} %")
fig.tight_layout()
save_figure(fig, paths.figures / "mm_mi_global_metrics_paired.png")
plt.show()

## 6. Même réseau ?

### Plafond 1 : moitiés aléatoires

Pour chaque sujet, ressemblance entre deux conditions = moyenne des 4 corrélations de Spearman croisées
(X-A/Y-A, X-A/Y-B, X-B/Y-A, X-B/Y-B) ; **plafond intra-condition** = moyenne de X-A/X-B et Y-A/Y-B. Toutes
les matrices comparées reposent sur n/2 epochs : le plafond mesure ce qu'on obtient quand rien ne change
sauf le bruit d'échantillonnage. Deux niveaux :

- **arêtes** : les 2 278 poids wPLI (triangle supérieur des matrices pondérées complètes, sans seuil) ;
- **profils de strength nodale** : les 68 strengths des graphes seuillés.

On rapporte le **ratio entre / intra** par sujet : médiane et IC à 95 % par bootstrap sur les sujets
(5 000 tirages). Un ratio proche de 1 signifie « aussi semblables que deux moitiés d'une même condition ».
Le Wilcoxon entre vs intra (Holm sur les 3 comparaisons) complète ; une p non significative ne suffit pas à
conclure à l'équivalence, d'où l'IC du ratio.

In [ ]:
sims = ta.similarity_table(cohort, DENSITIES)
sims.to_csv(TABLES / "mm_mi_similarity_by_subject.csv", index=False)
sim_summary = ta.similarity_summary(sims, N_BOOT, SEED)
sim_summary.to_csv(TABLES / "mm_mi_similarity_summary.csv", index=False)
main_sim = sim_summary[sim_summary.density.isna() | np.isclose(sim_summary.density, DENSITY)]
results["similarity_15"] = main_sim.round(6).to_dict(orient="records")
main_sim[
    [
        "band",
        "measure",
        "comparison",
        "median_between",
        "median_within",
        "median_ratio",
        "ratio_ci_low",
        "ratio_ci_high",
        "n_ratio_below_1",
        "W",
        "p_holm",
        "r_rb",
    ]
].round(4)

In [ ]:
fig, axes = plt.subplots(len(BANDS), 2, figsize=(13, 4.3 * len(BANDS)), squeeze=False)
comps = [f"{a} vs {b}" for a, b in ta.COMPARISONS]
for i, b in enumerate(BANDS):
    for j, (measure, dens, lab) in enumerate(
        [("edges", None, "arêtes (wPLI pondéré)"), ("strength", DENSITY, f"strength nodale ({round(DENSITY * 100)} %)")]
    ):
        ax = axes[i, j]
        sel = sims[(sims.band == b) & (sims.measure == measure)]
        if dens is not None:
            sel = sel[np.isclose(sel.density, dens)]
        data, pos, colors = [], [], []
        for k, comp in enumerate(comps):
            g = sel[sel.comparison == comp]
            data += [g.between.to_numpy(), g.within.to_numpy()]
            pos += [3 * k, 3 * k + 1]
            colors += ["C0", "C1"]
        bp = ax.boxplot(data, positions=pos, widths=0.8, patch_artist=True, showfliers=False)
        for patch, col in zip(bp["boxes"], colors, strict=True):
            patch.set_facecolor(col)
            patch.set_alpha(0.6)
        ax.set_xticks([3 * k + 0.5 for k in range(len(comps))], comps)
        ax.set_ylabel("ρ de Spearman")
        ax.set_title(f"{lab} — {b}", fontsize=10)
axes[0, 0].legend([bp["boxes"][0], bp["boxes"][1]], ["entre conditions", "intra-condition (plafond)"], fontsize=8)
fig.suptitle("Ressemblance par sujet, à nombre d'epochs égal (moitiés)")
fig.tight_layout()
save_figure(fig, paths.figures / "mm_mi_similarity_boxes.png")
plt.show()

### Plafond 2 : par run

Les moitiés aléatoires A et B mélangent des epochs des **mêmes runs**, alors que MM et MI viennent de runs
différents. Tout ce qui change d'un run à l'autre (dérive de l'état du sujet, de l'impédance des électrodes,
fatigue) abaisse donc la ressemblance « entre » sans abaisser le plafond 1, qui est optimiste.

Plafond 2 : une matrice wPLI par run (epochs tâche, même nombre d'epochs tiré au hasard dans les 6 runs ; un run
est gardé s'il a au moins 8 epochs tâche, et il faut 2 runs par condition, règle fixée avant de voir les
résultats ; S079 est écarté, d'où 104 sujets).

- **intra** = moyenne des similarités entre runs d'une même condition : MM 3-7, 3-11, 7-11 ; MI 4-8, 4-12, 8-12 ;
- **entre** = moyenne des 9 paires MM-run / MI-run ;
- même ratio entre / intra, IC 95 % bootstrap et Wilcoxon, sur les arêtes et la strength à 15 %.

Les runs MM et MI sont intercalés (3, 4, 7, 8, 11, 12) : les paires « entre » sont en moyenne plus proches
dans le temps que les paires « intra ». On refait donc la comparaison sur des paires **à distance appariée** :
on retire les 3 paires de runs adjacents (3-4, 7-8, 11-12) ; les 6 paires « entre » restantes ont la même
distance moyenne que les 6 paires « intra ».

In [ ]:
run_cohort = ta.load_run_matrices(paths.derived / "run_matrices_mm_mi.npz")
distances = pd.DataFrame(
    [
        {"paires": "intra (6)", "distance_moyenne_runs": ta.pair_distance(ta.INTRA_RUN_PAIRS)},
        {"paires": "entre, toutes (9)", "distance_moyenne_runs": ta.pair_distance(ta.BETWEEN_RUN_PAIRS)},
        {
            "paires": "entre, distance appariée (6)",
            "distance_moyenne_runs": ta.pair_distance(ta.MATCHED_BETWEEN_RUN_PAIRS),
        },
    ]
)
print(
    f"{len(run_cohort.subjects)} sujets ; epochs par run : médiane {np.median(run_cohort.n_per_run):.0f} "
    f"({run_cohort.n_per_run.min()}-{run_cohort.n_per_run.max()})"
)
display(distances.round(2))
run_sims = ta.run_similarity_table(run_cohort, DENSITY)
run_sims.to_csv(TABLES / "mm_mi_run_similarity_by_subject.csv", index=False)
run_summary = ta.similarity_summary(run_sims, N_BOOT, SEED)
run_summary.to_csv(TABLES / "mm_mi_run_similarity_summary.csv", index=False)
results["run_ceiling"] = {
    "n_subjects": len(run_cohort.subjects),
    "n_per_run_median": float(np.median(run_cohort.n_per_run)),
    "distances": distances.to_dict(orient="records"),
    "summary": run_summary.round(6).to_dict(orient="records"),
}
cols = [
    "band",
    "measure",
    "comparison",
    "median_between",
    "median_within",
    "median_ratio",
    "ratio_ci_low",
    "ratio_ci_high",
    "n_ratio_below_1",
    "W",
    "p_holm",
    "r_rb",
]
run_summary[cols].round(4)

**Similarité en fonction de la distance entre runs** (médiane des sujets). Si la ressemblance baisse avec la
distance, l'écart entre les plafonds 1 et 2 vient en partie du temps ; la comparaison à distance égale isole
l'effet de la condition.

In [ ]:
profile = ta.run_pair_profile(run_cohort, DENSITY)
profile.to_csv(TABLES / "mm_mi_run_pair_profile.csv", index=False)
prof = profile.groupby(["band", "measure", "type", "distance"]).similarity.median().unstack(["type", "distance"])
results["run_pair_profile"] = {
    f"{b} {m}": {f"{t} d={d}": round(float(v), 4) for (t, d), v in row.items()} for (b, m), row in prof.iterrows()
}
display(prof.round(3))
fig, axes = plt.subplots(len(BANDS), 2, figsize=(11, 3.8 * len(BANDS)), squeeze=False)
for i, b in enumerate(BANDS):
    for j, (measure, lab) in enumerate([("edges", "arêtes"), ("strength", f"strength {round(DENSITY * 100)} %")]):
        ax = axes[i, j]
        sub = profile[(profile.band == b) & (profile.measure == measure)]
        for kind, color in (("entre", "C0"), ("intra", "C1")):
            g = sub[sub.type == kind].groupby("distance").similarity
            med, q1, q3 = g.median(), g.quantile(0.25), g.quantile(0.75)
            ax.errorbar(
                med.index,
                med,
                yerr=[med - q1, q3 - med],
                fmt="o-",
                color=color,
                capsize=3,
                label=f"{kind} (médiane, IQR)",
            )
        ax.set(xlabel="distance entre runs (numéros de run)", ylabel="ρ de Spearman", title=f"{lab} — {b}")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
save_figure(fig, paths.figures / "mm_mi_run_ceiling_distance.png")
plt.show()

## 7. Hubs et région sensorimotrice

Hubs = 7 régions de plus forte strength moyenne (sur les sujets) dans chaque condition ; recouvrement par
l'indice de Jaccard. Puis : la strength moyenne de la ROI sensorimotrice (precentral, postcentral, paracentral,
deux côtés) augmente-t-elle en MM et en MI par rapport à REST ? Wilcoxon apparié bilatéral, Holm sur les 2
tests de chaque bande.

In [ ]:
strengths = ta.subject_strengths(cohort, DENSITY)  # (sujets, conditions, bandes, régions)
hub_rows, roi_rows = [], []
hubs = {}
for k, b in enumerate(BANDS):
    mean_s = strengths[:, :, k].mean(axis=0)
    for c in CONDS:
        hubs[c, b] = ta.top_hubs(mean_s[CONDS.index(c)], atlas, 7)
    for x, y in ta.COMPARISONS:
        hub_rows.append(
            {
                "band": b,
                "comparison": f"{x} vs {y}",
                "jaccard": ta.jaccard(hubs[x, b], hubs[y, b]),
                "common": ", ".join(sorted(set(hubs[x, b]) & set(hubs[y, b]))),
            }
        )
    roi = ta.roi_values(strengths[:, :, k], atlas)  # (sujets, conditions)
    block = [
        {"band": b, "comparison": f"{c} vs REST"} | ta.paired_test(roi[:, CONDS.index(c)], roi[:, CONDS.index("REST")])
        for c in ("MM", "MI")
    ]
    for row, p in zip(block, ta.holm([r["p"] for r in block]), strict=True):
        row["p_holm"] = float(p)
    roi_rows += block
hubs_df = pd.DataFrame([{"band": b, "condition": c, "hubs": ", ".join(h)} for (c, b), h in hubs.items()])
hub_overlap = pd.DataFrame(hub_rows)
roi_tests = pd.DataFrame(roi_rows)
hubs_df.to_csv(TABLES / "mm_mi_hubs.csv", index=False)
roi_tests.to_csv(TABLES / "mm_mi_roi_strength_tests.csv", index=False)
results["hubs"] = hubs_df.to_dict(orient="records")
results["hub_overlap"] = hub_overlap.round(4).to_dict(orient="records")
results["roi_strength_tests"] = roi_tests.round(6).to_dict(orient="records")
display(hubs_df)
display(hub_overlap.round(3))
roi_tests.round(4)

**Contrôle de confusion puissance / connectivité.** La puissance alpha/bêta baisse davantage en MM qu'en MI
sur la ROI (ERD). Un rapport signal sur bruit plus faible peut à lui seul modifier un wPLI. Sujet par sujet,
on corrèle (Spearman) la différence de puissance de la ROI (MM − MI, log10, mêmes epochs égalisées) avec la
différence de strength de la ROI (MM − MI). Le résultat est rapporté, pas corrigé.

In [ ]:
confound = pd.DataFrame(
    [
        {"band": b} | ta.power_strength_confound(cohort.power[:, :, k], strengths[:, :, k], atlas)
        for k, b in enumerate(BANDS)
    ]
)
confound.to_csv(TABLES / "mm_mi_power_strength_confound.csv", index=False)
results["power_strength_confound"] = confound.round(6).to_dict(orient="records")
confound.round(4)

## 8. Communautés

Accord entre les partitions de Louvain des graphes de groupe (NMI, ARI : 1 = partitions identiques). Le
**module sensorimoteur** d'une condition est la communauté qui contient le plus de régions de la ROI ; on
liste ses régions et leur intersection entre MM et MI.

In [ ]:
comm_rows, modules = [], {}
for b in BANDS:
    for c in CONDS:
        modules[c, b] = ta.sensorimotor_module(group[c, b].metrics.community, atlas)
    for x, y in ta.COMPARISONS:
        common = sorted(set(modules[x, b]) & set(modules[y, b]))
        comm_rows.append(
            {"band": b, "comparison": f"{x} vs {y}"}
            | ta.partition_similarity(group[x, b].metrics.community, group[y, b].metrics.community)
            | {
                "module_jaccard": ta.jaccard(modules[x, b], modules[y, b]),
                "n_common": len(common),
                "common_module": ", ".join(common),
            }
        )
communities = pd.DataFrame(comm_rows)
communities.to_csv(TABLES / "mm_mi_communities.csv", index=False)
results["communities"] = communities.round(4).to_dict(orient="records")
results["sensorimotor_modules"] = {f"{c}_{b}": m for (c, b), m in modules.items()}
for (c, b), m in modules.items():
    print(f"{b} {c} : module sensorimoteur de {len(m)} régions, {sum(r in MOTOR_ROI for r in m)}/6 de la ROI")
communities.round(3)

# Partie C — Robustesse

Points 5 et 6 refaits aux seuils 10 % et 20 %, en alpha et en bêta. Chaque cellule donne le sens de l'effet
(« + » : la première condition est plus haute, ou la ressemblance entre conditions dépasse le plafond ; « − » :
l'inverse), la p de Holm (« * » si < 0.05) et ✓ si la conclusion est la même qu'au seuil de 15 % de la même
bande. Le point 6 utilise ici le plafond 1 (moitiés), seul plafond calculé à chaque seuil. La similarité des
arêtes ne dépend pas du seuil (matrices pondérées complètes) : sa ligne est la même pour les trois seuils.

Deux définitions de « la conclusion tient » :
- **stricte** : même sens et même statut de significativité ;
- **clémente** : un résultat significatif doit le rester dans le même sens ; un résultat non significatif qui
  reste non significatif tient, quel que soit son signe (un effet nul n'a pas de sens à conserver).

Le tableau affiché utilise la définition clémente.

In [ ]:
robust_strict = ta.robustness_table(tests, sim_summary)
robust = ta.robustness_table(tests, sim_summary, lenient=True)
robust_strict.to_csv(TABLES / "mm_mi_robustness_strict.csv")
robust.to_csv(TABLES / "mm_mi_robustness.csv")


def holds_fraction(table: pd.DataFrame) -> float:
    return float(table.apply(lambda col: col.str.endswith("✓")).to_numpy().mean())


results["robustness"] = robust.to_dict(orient="index")
results["robustness_holds_fraction"] = {"strict": holds_fraction(robust_strict), "lenient": holds_fraction(robust)}
print(
    f"Conclusions qui tiennent : {holds_fraction(robust):.0%} (clémente), {holds_fraction(robust_strict):.0%} (stricte)"
)
robust

In [ ]:
(paths.root / "reports" / "mm_vs_mi_results.json").write_text(
    json.dumps(results, indent=2, ensure_ascii=False, default=str)
)
print("Figures :", sorted(p.name for p in paths.figures.glob("mm_mi_*.png")))

# Limites

- **Module sensorimoteur commun latéralisé à droite** : il regroupe precentral, postcentral et paracentral
  droits avec des régions pariéto-temporo-occipitales droites ; les régions sensorimotrices gauches tombent
  dans d'autres communautés.
- **ERD en MI plus forte à gauche**, alors que les essais gauche et droite sont regroupés (notebook 02 et
  `reports/validation_task_cohort.json`). Observation non expliquée, non investiguée.
- **Confusion puissance / connectivité en alpha** : la différence de strength de la ROI MM − MI suit la
  différence de puissance (ρ de Spearman = 0.29, p = 0.003). Une baisse de puissance (ERD) réduit le rapport
  signal sur bruit et peut, à elle seule, modifier un wPLI. Rapporté, pas corrigé.
- **Template fsaverage sans IRM individuelle** : modèle direct commun à tous les sujets ; la localisation
  des sources et la fuite entre régions voisines restent approximatives (régions de 68 parcelles, eLORETA).
- **Biais run-à-run du plafond aléatoire** : les moitiés A et B mélangent les mêmes runs, alors que MM et MI
  viennent de runs différents ; le plafond 1 est optimiste. Le plafond 2 (par run) le corrige, mais ses
  paires « entre » sont plus proches dans le temps ; seule la comparaison à distance appariée équilibre les deux.

# Conclusion

**En grande partie le même réseau.** MM et MI partagent les mêmes hubs (Jaccard des 7 hubs = 1.00 en alpha,
0.75 en bêta) et le même module sensorimoteur (Jaccard 0.75 et 0.94), et ne diffèrent sur aucune métrique
globale au seuil de 15 % (p de Holm ≥ 0.078). Face au plafond par run, qui intègre la variabilité d'un run à
l'autre, la ressemblance des arêtes MM–MI atteint 96 % du plafond en alpha et 99.5 % en bêta (IC 95 %
98.8-100.4 %) quand on prend les 9 paires de runs, mais ces paires sont plus proches dans le temps (3.9 contre
5.3 runs d'écart) et la ressemblance diminue avec la distance entre runs. À distance appariée (5.3 runs des
deux côtés), MM–MI reste sous le plafond : 95 % en alpha (IC 92-96 %) et 97 % en bêta (97-98 %) pour les
arêtes, ratio < 1 chez 78 et 88 sujets sur 104 (p de Holm ≤ 6e-9), proche des 94 et 96 % du plafond
aléatoire. Cet écart lié à la condition est faible, du même ordre que l'écart de chaque tâche avec le repos,
et l'effet de tâche le plus net est commun aux deux : en bêta, la strength de la ROI sensorimotrice augmente
autant en MM qu'en MI par rapport au repos (r = 0.45 et 0.48, p de Holm ≤ 5e-5). Mouvement réel et imaginé
reposent donc sur un même réseau, modulé faiblement mais de façon reproductible par la condition (94 % des
conclusions tiennent aux seuils de 10 et 20 %), sous les réserves de la cellule « Limites ».
